# Genie360 — 03: Rewrite Pipeline Test

End-to-end test for the rewrite pipeline:
- Takes sample queries (real or synthetic)
- Runs the full rule → LLM → validation pipeline
- Displays side-by-side before/after with confidence scores
- Shows EXPLAIN plan comparisons

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from genie360.modules.anti_pattern_detection import run_anti_pattern_suite
from genie360.modules.sql_rewrite_engine import (
    apply_rule_based_rewrites,
    calculate_rewrite_impact,
    serialize_ast_to_sql,
)
from genie360.modules.llm_refinement import (
    build_llm_rewrite_prompt,
    invoke_llm_rewrite,
    parse_llm_rewrite_response,
    score_rewrite_confidence,
)
from genie360.pipeline import load_genie360_config

config = load_genie360_config(project_root / "genie360" / "config.yml")
print("✅ All modules imported")

✅ All modules imported


## 1. Test Queries

Representative problematic Genie-generated queries for pipeline testing.

In [2]:
TEST_QUERIES = [
    {
        "name": "Full table scan with SELECT *",
        "sql": "SELECT * FROM catalog.schema.orders",
    },
    {
        "name": "Missing LIMIT on large table",
        "sql": "SELECT order_id, customer_id, amount, status FROM catalog.schema.orders WHERE amount > 100",
    },
    {
        "name": "COUNT DISTINCT on high-cardinality column",
        "sql": "SELECT region, COUNT(DISTINCT customer_id) AS unique_customers FROM catalog.schema.orders GROUP BY region",
    },
    {
        "name": "Cross join risk",
        "sql": "SELECT a.order_id, b.name FROM catalog.schema.orders a CROSS JOIN catalog.schema.customers b WHERE a.amount > 1000",
    },
    {
        "name": "Complex subquery",
        "sql": "SELECT customer_id, total FROM (SELECT customer_id, SUM(amount) as total FROM catalog.schema.orders GROUP BY customer_id) t WHERE total > (SELECT AVG(amount) * 10 FROM catalog.schema.orders)",
    },
]

## 2. Rule-Based Rewrite Pipeline

In [3]:
import pandas as pd

rewrite_results = []

for test in TEST_QUERIES:
    # Step 1: Detect anti-patterns
    report = run_anti_pattern_suite(test["sql"])

    # Step 2: Apply rule-based rewrites
    rewritten_sql, rules_applied = apply_rule_based_rewrites(test["sql"])

    # Step 3: Calculate impact
    impact = calculate_rewrite_impact(test["sql"], rewritten_sql)

    rewrite_results.append({
        "Query": test["name"],
        "Issues Found": report.total_issues,
        "Max Severity": report.max_severity.value,
        "Rules Applied": ", ".join(rules_applied) if rules_applied else "(none)",
        "Issues Fixed": impact["issues_fixed"],
        "Improvement %": f"{impact['estimated_improvement_pct']}%",
    })

print("\n📋 Rule-Based Rewrite Results:\n")
pd.DataFrame(rewrite_results)


📋 Rule-Based Rewrite Results:



,Query,Issues Found,Max Severity,Rules Applied,Issues Fixed,Improvement %
0,Full table scan with SELECT *,3,HIGH,LIMIT_INJECTION,1,33.3%
1,Missing LIMIT on large table,1,HIGH,LIMIT_INJECTION,1,100.0%
2,COUNT DISTINCT on high-cardinality column,1,HIGH,COUNT_DISTINCT_TO_APPROX,0,0.0%
3,Cross join risk,2,CRITICAL,LIMIT_INJECTION,1,50.0%
4,Complex subquery,0,INFO,(none),0,0.0%


## 3. Side-by-Side SQL Comparison

In [4]:
for test in TEST_QUERIES:
    rewritten, rules = apply_rule_based_rewrites(test["sql"])
    print(f"\n{'='*70}")
    print(f"📝 {test['name']}")
    print(f"{'='*70}")
    print(f"\n🔴 ORIGINAL:\n{test['sql']}")
    print(f"\n🟢 REWRITTEN:\n{rewritten}")
    print(f"\n📌 Rules applied: {', '.join(rules) if rules else '(no changes)'}")


📝 Full table scan with SELECT *

🔴 ORIGINAL:
SELECT * FROM catalog.schema.orders

🟢 REWRITTEN:
SELECT
  *
FROM catalog.schema.orders
LIMIT 10000

📌 Rules applied: LIMIT_INJECTION

📝 Missing LIMIT on large table

🔴 ORIGINAL:
SELECT order_id, customer_id, amount, status FROM catalog.schema.orders WHERE amount > 100

🟢 REWRITTEN:
SELECT
  order_id,
  customer_id,
  amount,
  status
FROM catalog.schema.orders
WHERE
  amount > 100
LIMIT 10000

📌 Rules applied: LIMIT_INJECTION

📝 COUNT DISTINCT on high-cardinality column

🔴 ORIGINAL:
SELECT region, COUNT(DISTINCT customer_id) AS unique_customers FROM catalog.schema.orders GROUP BY region

🟢 REWRITTEN:
SELECT
  region,
  APPROX_COUNT_DISTINCT(customer_id) AS unique_customers
FROM catalog.schema.orders
GROUP BY
  region /* NOTE: Using approximate count (~2% error, 10-100x faster) */

📌 Rules applied: COUNT_DISTINCT_TO_APPROX

📝 Cross join risk

🔴 ORIGINAL:
SELECT a.order_id, b.name FROM catalog.schema.orders a CROSS JOIN catalog.schema.custom

## 4. LLM Refinement (Optional — requires Databricks Model Serving)

Uncomment and run the cell below if you have access to a Databricks LLM endpoint.

In [8]:
LLM_ENABLED = True  # Set to True to test LLM refinement

if LLM_ENABLED:
    for test in TEST_QUERIES[:2]:  # Test on first 2 queries
        report = run_anti_pattern_suite(test["sql"])
        rewritten, rules = apply_rule_based_rewrites(test["sql"])

        prompt = build_llm_rewrite_prompt(
            original_sql=test["sql"],
            rule_rewritten_sql=rewritten if rules else None,
            anti_pattern_report=report,
        )

        llm_response = invoke_llm_rewrite(prompt)
        llm_sql, explanation, confidence = parse_llm_rewrite_response(llm_response)

        print(f"\n{'='*70}")
        print(f"🤖 LLM Refinement: {test['name']}")
        print(f"{'='*70}")
        print(f"Rule-based: {rewritten[:100]}...")
        print(f"LLM-refined: {llm_sql[:100] if llm_sql else 'N/A'}...")
        print(f"Confidence: {confidence}")


🤖 LLM Refinement: Full table scan with SELECT *
Rule-based: SELECT
  *
FROM catalog.schema.orders
LIMIT 10000...
LLM-refined: SELECT
  order_id,
  customer_id,
  order_date,
  total_amount,
  status,
  created_at,
  updated_at...
Confidence: 0.7

🤖 LLM Refinement: Missing LIMIT on large table
Rule-based: SELECT
  order_id,
  customer_id,
  amount,
  status
FROM catalog.schema.orders
WHERE
  amount > 100...
LLM-refined: SELECT
  order_id,
  customer_id,
  amount,
  status
FROM catalog.schema.orders
WHERE
  amount > 100...
Confidence: 0.7


In [6]:
# Confidence scoring (without spark for offline testing)
for test in TEST_QUERIES:
    report = run_anti_pattern_suite(test["sql"])
    confidence = score_rewrite_confidence(
        anti_pattern_report=report,
        explain_plan_comparison=None,
        semantic_equivalence_result=None,
    )
    print(f"{test['name'][:40]:40s} → composite={confidence.composite_score:.3f} "
          f"(rule={confidence.rule_coverage_score:.2f}, "
          f"explain={confidence.explain_plan_score:.2f}, "
          f"semantic={confidence.semantic_equivalence_score:.2f})")

print("\n🎯 Rewrite Pipeline Test Complete!")

Full table scan with SELECT *            → composite=0.520 (rule=0.60, explain=0.50, semantic=0.50)
Missing LIMIT on large table             → composite=0.440 (rule=0.20, explain=0.50, semantic=0.50)
COUNT DISTINCT on high-cardinality colum → composite=0.440 (rule=0.20, explain=0.50, semantic=0.50)
Cross join risk                          → composite=0.480 (rule=0.40, explain=0.50, semantic=0.50)
Complex subquery                         → composite=0.420 (rule=0.10, explain=0.50, semantic=0.50)

🎯 Rewrite Pipeline Test Complete!
